# homr — Türk müziği (makam) ince ayarı

SymbTr'den üretilmiş porte verisiyle homr'un **arıza (lift)** ve **ritim** başlarını ince ayarlar.

**Runtime → Change runtime type → GPU** seç. A100 / L4 / V100 ideal (bf16 destekler).

Hücreler sırayla çalıştırılır. 6. hücre kısa deneme, 7. hücre gerçek eğitim.

In [ ]:
#@title 1. GPU uygun mu?
import subprocess, torch
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'],
                     capture_output=True, text=True).stdout)
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('\nGPU YOK -> Runtime > Change runtime type > GPU')
else:
    bf16 = torch.cuda.is_bf16_supported()
    print('kart:', torch.cuda.get_device_name(0), '| bf16:', bf16)
    print('\nHazir.' if bf16 else
          '\nUYARI: bu kart bf16 desteklemiyor (T4 gibi).'
          ' 6. ve 7. hucrelere --fp32 ekle, yoksa egitim patlar.')

In [ ]:
#@title 2. Depoyu çek
REPO_URL = 'https://github.com/mtalhabalci/homr_tmn.git'  #@param {type:"string"}
BRANCH   = 'makam-finetune'  #@param {type:"string"}

import os, shutil
if os.path.isdir('/content/homr'):
    shutil.rmtree('/content/homr')
!git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/homr
os.chdir('/content/homr')
!git log --oneline -1

In [ ]:
#@title 3. Bagimliliklar
# transformers 5.x TrainingArguments'tan warmup_ratio'yu kaldirdi;
# depo 4.x bekliyor (pyproject.toml: ^4.53.2), o yuzden sabitliyoruz.
!pip install -q 'transformers>=4.53.2,<5' albumentations editdistance \
                'musicxml==1.4' x-transformers onnxruntime pymupdf

import transformers, inspect
from transformers import TrainingArguments
print('transformers', transformers.__version__)
need = ['warmup_ratio', 'eval_strategy', 'torch_compile', 'bf16']
have = inspect.signature(TrainingArguments.__init__).parameters
for n in need:
    print('  %-16s %s' % (n, 'var' if n in have else 'YOK -> surum uyumsuz'))


## 4. Veri

Porte resimleri git'e sigmayacak kadar buyuk (~250 MB), o yuzden Drive'dan geliyor.

Drive'da **homr_makam** adli bir klasor ac ve **symbtr_veri.tar.gz** dosyasini
icine koy. Arsiv Colab'in *yerel diskine* aciliyor, egitim oradan okuyor -
Drive'dan dogrudan okumak cok yavas olurdu.

Arsivi yeniden uretmek gerekirse, yerel makinede:
`python -m notebooks.package_dataset`


In [ ]:
#@title 4. Veriyi Drive'dan aç
from google.colab import drive
drive.mount('/content/drive')

import os, tarfile, time
os.chdir('/content/homr')
archive = '/content/drive/MyDrive/homr_makam/symbtr_veri.tar.gz'
assert os.path.exists(archive), f'{archive} yok - once Drive a yukle'
t0 = time.time()
with tarfile.open(archive) as t:
    t.extractall('/content/homr')
print('acildi (%.0f sn)' % (time.time() - t0))

In [ ]:
#@title 5. Sözlük ve veri kontrolü (eğitimden önce)
import os, sys
os.chdir('/content/homr'); sys.path.insert(0, '/content/homr')
from homr.transformer.vocabulary import Vocabulary

v = Vocabulary()
print('rhythm %d | lift %d | pitch %d' % (len(v.rhythm), len(v.lift), len(v.pitch)))
print('ariza jetonlari  :', [k for k in v.lift if k[:5] in ('sharp', 'flat')])
print('keyAccidental    :', 'keyAccidental' in v.rhythm)
print('timeSignature_9/8:', 'timeSignature_9/8' in v.rhythm)
print()
for name in ('train', 'val', 'test'):
    path = f'datasets/SymbTr-2.0.0/index_{name}.txt'
    print('%-6s %6d porte' % (name, sum(1 for _ in open(path))))

bad = 0
with open('datasets/SymbTr-2.0.0/index_train.txt') as f:
    rows = [l for l in f if l.strip()][:300]
for row in rows:
    image, tokens = row.strip().split(',')
    if not (os.path.exists(image) and os.path.exists(tokens)):
        bad += 1
        continue
    for line in open(tokens, encoding='utf-8'):
        p = line.split()
        if len(p) == 5 and (p[0] not in v.rhythm or p[2] not in v.lift):
            bad += 1
print('\n300 ornek kontrol edildi, sorunlu:', bad)

In [ ]:
#@title 6. KISA DENEME — 1 epoch, 400 porte
# Ilk calistirmada hazir checkpoint'i indirir (~279 MB).
# bf16 desteklemeyen kartta sona --fp32 ekle.
!cd /content/homr && python -m training.transformer.train --fine --epochs 1 --limit 400

In [ ]:
#@title 7. GERÇEK EĞİTİM — 15 epoch, tüm veri
# Checkpoint'ler ve bitmiş model Drive'a yazılır, oturum ölse de kaybolmaz.
# Kesilirse: aynı komuta --resume checkpoint-XXXX ekleyip tekrar çalıştır.
LOG = '/content/drive/MyDrive/homr_makam/log_full.txt'
!cd /content/homr && python -m training.transformer.train --fine \
    --work-dir /content/drive/MyDrive/homr_makam 2>&1 | tee {LOG}


In [ ]:
#@title 8. Değerlendirme — sınıf sınıf
import glob, os
model = max(glob.glob('/content/drive/MyDrive/homr_makam/*.pth'), key=os.path.getmtime)
print('Model:', model)
OUT = '/content/drive/MyDrive/homr_makam/eval_full.txt'
!cd /content/homr && python -m training.evaluate_makam --checkpoint "{model}" 2>&1 | tee {OUT}
